[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C17_Classical_NLP_Course/04_crf_sequence/04_crf_sequence.ipynb)

# 04 · CRF 与结构化预测（纯 numpy 从零）

目标：**不调任何 CRF 库**，从零实现**特征函数**、线性链 CRF 的**对数配分函数**（=CRF 版前向）、**Viterbi 解码**、**结构化感知机**，并对拍暴力枚举、在 NER 上随训练把 F1 提上去。

路线：特征模板 → 序列打分 → 暴力 $Z$ → **对数前向算 $\log Z$(对拍暴力)** → **Viterbi(对拍暴力)** → **结构化感知机训练** → ✏️ 练习 → 📖 答案 → 🧪 真实 NER 胶囊。

> 心智模型：**CRF = HMM 的 DP 骨架(前向/Viterbi) + 判别式打分 + 全局归一**。配分函数就是前向，解码就是 Viterbi。

## 1 · 特征函数与序列打分

线性链 CRF 给标签序列打分：$\mathrm{score}(y,x)=\sum_t[\text{emit}(y_t,x,t) + \text{trans}(y_{t-1},y_t)]$。
用最简结构：发射得分 = 权重表 `W_emit[tag, word]`，转移得分 = `W_trans[prev_tag, tag]`。

In [ ]:
import numpy as np
import itertools
rng = np.random.default_rng(0)

tags = ['O', 'B-PER', 'I-PER']
T2I = {t: i for i, t in enumerate(tags)}
K_t = len(tags)
vocab = ['<s>', 'mr', 'john', 'smith', 'ran', 'fast']
W2I = {w: i for i, w in enumerate(vocab)}
Vn = len(vocab)

# 随机初始化权重(小值)
W_emit = rng.standard_normal((K_t, Vn)) * 0.1     # 发射: tag x word
W_trans = rng.standard_normal((K_t, K_t)) * 0.1   # 转移: prev_tag x tag
START = T2I['O']                                   # 句首前驱视为 O

def seq_score(words, tag_seq, W_emit, W_trans):
    '''一条标签序列的总得分。'''
    o = [W2I[w] for w in words]; y = [T2I[t] for t in tag_seq]
    s = W_emit[y[0], o[0]] + W_trans[START, y[0]]
    for t in range(1, len(o)):
        s += W_emit[y[t], o[t]] + W_trans[y[t-1], y[t]]
    return s

words = ['mr', 'john', 'smith']
sc = seq_score(words, ['O', 'B-PER', 'I-PER'], W_emit, W_trans)
print('score([O, B-PER, I-PER]) =', round(sc, 4))
assert np.isfinite(sc)
print('✅ 特征打分就绪：发射 + 转移 的加权和')

## 2 · 暴力枚举配分函数 Z（ground truth）

$Z(x)=\sum_{y'}\exp(\mathrm{score}(y',x))$ 对所有 $K^T$ 条标签序列求和。小规模可枚举——验证对数前向的**绝对参考**。

In [ ]:
def brute_force_logZ(words, W_emit, W_trans):
    T = len(words)
    scores = [seq_score(words, list(y), W_emit, W_trans)
              for y in itertools.product(tags, repeat=T)]
    scores = np.array(scores)
    m = scores.max()
    return m + np.log(np.sum(np.exp(scores - m)))     # log-sum-exp

logZ_bf = brute_force_logZ(words, W_emit, W_trans)
print(f'log Z 暴力枚举 {K_t**len(words)} 条序列 = {logZ_bf:.6f}')
assert np.isfinite(logZ_bf)
print('✅ 暴力 log Z 就绪 —— 对数前向必须复现它')

## 3 · 对数前向算 log Z（对拍暴力）

$\alpha_t(j)=\text{logsumexp}_i[\alpha_{t-1}(i)+\text{trans}(i,j)]+\text{emit}(j,x,t)$，$\log Z=\text{logsumexp}_j\alpha_T(j)$。
和 HMM 前向同构——只是「乘概率」换成「加得分」。

In [ ]:
def logsumexp(x, axis=None):
    x = np.asarray(x, float)
    if axis is None:
        m = np.max(x); return float(m + np.log(np.sum(np.exp(x - m))))
    m = np.max(x, axis=axis, keepdims=True)
    return np.squeeze(m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True)), axis=axis)

def forward_logZ(words, W_emit, W_trans):
    o = [W2I[w] for w in words]; T = len(o)
    alpha = W_emit[:, o[0]] + W_trans[START, :]        # alpha_0(j)
    for t in range(1, T):
        new = np.zeros(K_t)
        for j in range(K_t):
            new[j] = logsumexp(alpha + W_trans[:, j]) + W_emit[j, o[t]]
        alpha = new
    return logsumexp(alpha)

logZ = forward_logZ(words, W_emit, W_trans)
print(f'对数前向 log Z = {logZ:.6f}')
print(f'暴力     log Z = {logZ_bf:.6f}')
assert np.allclose(logZ, logZ_bf, atol=1e-10), '对数前向必须对拍暴力'
print('✅ 配分函数 = CRF 版前向算法, 对拍暴力到 1e-10')

## 4 · CRF 概率与归一性

$P(y|x)=\exp(\mathrm{score}(y,x)-\log Z)$。**健全性检验**：对所有 $y$ 求 $P(y|x)$ 之和应为 1。

In [ ]:
def crf_logprob(words, tag_seq, W_emit, W_trans):
    return seq_score(words, tag_seq, W_emit, W_trans) - forward_logZ(words, W_emit, W_trans)

# 所有标签序列的概率应和为 1
total_p = 0.0
for y in itertools.product(tags, repeat=len(words)):
    total_p += np.exp(crf_logprob(words, list(y), W_emit, W_trans))
print('Σ_y P(y|x) =', round(total_p, 9))
assert abs(total_p - 1.0) < 1e-9, 'CRF 概率必须归一'
# 概率都在 (0,1]
p1 = np.exp(crf_logprob(words, ['O','B-PER','I-PER'], W_emit, W_trans))
assert 0 < p1 <= 1
print('✅ CRF 是合法概率分布 (Σ_y P=1) —— 全局归一的体现')

## 5 · Viterbi 解码（对拍暴力）

解码 $\arg\max_y\mathrm{score}(y,x)$（不需 $Z$）。把前向的 logsumexp 换成 max + 回溯指针——又是 sum→max 对偶。

In [ ]:
def viterbi_crf(words, W_emit, W_trans):
    o = [W2I[w] for w in words]; T = len(o)
    delta = W_emit[:, o[0]] + W_trans[START, :]
    psi = np.zeros((T, K_t), dtype=int)
    for t in range(1, T):
        new = np.zeros(K_t)
        for j in range(K_t):
            sc = delta + W_trans[:, j]
            psi[t, j] = int(np.argmax(sc))
            new[j] = sc[psi[t, j]] + W_emit[j, o[t]]
        delta = new
    last = int(np.argmax(delta)); path = [last]
    for t in range(T - 1, 0, -1):
        path.append(psi[t, path[-1]])
    path.reverse()
    return [tags[i] for i in path], float(delta[last])

def brute_force_best_tags(words, W_emit, W_trans):
    best, bs = None, -np.inf
    for y in itertools.product(tags, repeat=len(words)):
        s = seq_score(words, list(y), W_emit, W_trans)
        if s > bs: bs, best = s, list(y)
    return best, bs

vpath, vscore = viterbi_crf(words, W_emit, W_trans)
bpath, bscore = brute_force_best_tags(words, W_emit, W_trans)
print('Viterbi:', vpath, round(vscore, 4))
print('暴力   :', bpath, round(bscore, 4))
assert vpath == bpath and np.allclose(vscore, bscore, atol=1e-10)
assert np.allclose(vscore, seq_score(words, vpath, W_emit, W_trans), atol=1e-10)
print('✅ Viterbi == 暴力最优, 得分==该序列得分 —— 解码正确')

## 6 · 结构化感知机训练

Collins 2002：Viterbi 解码错了就「真值特征 +1、预测特征 −1」。不需配分函数，实现极简。
在一个玩具 NER 上训练，看准确率随训练上升。

In [ ]:
def feature_counts(words, tag_seq):
    '''返回该 (words, tags) 的发射/转移特征计数 (与权重同形)。'''
    o = [W2I[w] for w in words]; y = [T2I[t] for t in tag_seq]
    fe = np.zeros((K_t, Vn)); ft = np.zeros((K_t, K_t))
    fe[y[0], o[0]] += 1; ft[START, y[0]] += 1
    for t in range(1, len(o)):
        fe[y[t], o[t]] += 1; ft[y[t-1], y[t]] += 1
    return fe, ft

def train_perceptron(train, epochs=20):
    We = np.zeros((K_t, Vn)); Wt = np.zeros((K_t, K_t))
    We_sum = np.zeros_like(We); Wt_sum = np.zeros_like(Wt); n = 0
    for ep in range(epochs):
        for words_, gold in train:
            pred, _ = viterbi_crf(words_, We, Wt)
            if pred != gold:
                fe_g, ft_g = feature_counts(words_, gold)
                fe_p, ft_p = feature_counts(words_, pred)
                We += fe_g - fe_p; Wt += ft_g - ft_p   # 奖励真值, 惩罚预测
            We_sum += We; Wt_sum += Wt; n += 1
    return We_sum / n, Wt_sum / n                       # averaged perceptron

train_data = [
    (['mr', 'john', 'smith'], ['O', 'B-PER', 'I-PER']),
    (['john', 'ran', 'fast'], ['B-PER', 'O', 'O']),
    (['mr', 'smith', 'ran'], ['O', 'B-PER', 'O']),
]
We, Wt = train_perceptron(train_data, epochs=20)
# 训练后, 在训练集上解码应基本正确
correct = sum(viterbi_crf(w, We, Wt)[0] == g for w, g in train_data)
print(f'训练后训练集正确句数: {correct}/{len(train_data)}')
assert correct == len(train_data), '结构化感知机应学会拟合训练集'
print('✅ 结构化感知机: Viterbi 解码 + 错则调权重, 学会了 NER 模式')

---
## ✏️ 练习 1：特征模板

实现 `is_capitalized_feature(word)`：返回该词是否首字母大写（NER 的强信号特征）。再实现 `suffix_feature(word, k)`：返回后 `k` 个字符（如 -ton/-ville 暗示地名）。

In [ ]:
def is_capitalized_feature(word):
    # TODO: 返回 1 if 首字母大写 else 0
    raise NotImplementedError

def suffix_feature(word, k=3):
    # TODO: 返回 word 的后 k 个字符
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert is_capitalized_feature('London') == 1
assert is_capitalized_feature('the') == 0
assert is_capitalized_feature('') == 0
assert suffix_feature('Washington', 3) == 'ton'
assert suffix_feature('go', 3) == 'go', '短词返回整个词'
print('✅ 练习 1 通过：大写与后缀特征')

## ✏️ 练习 2：对数前向（配分函数）

实现 `logZ_forward(words, W_emit, W_trans)`：对数空间前向算 $\log Z$。（用给定 `logsumexp`、`W2I`、`START`、`K_t`。）

In [ ]:
def logZ_forward(words, W_emit, W_trans):
    # TODO: alpha = W_emit[:,o0] + W_trans[START]; 递推 logsumexp; 返回 logsumexp(alpha)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
lz = logZ_forward(words, W_emit, W_trans)
assert np.allclose(lz, brute_force_logZ(words, W_emit, W_trans), atol=1e-10), '应对拍暴力'
# 另一句也要对
w2 = ['john', 'ran', 'fast']
assert np.allclose(logZ_forward(w2, W_emit, W_trans),
                   brute_force_logZ(w2, W_emit, W_trans), atol=1e-10)
print('✅ 练习 2 通过：对数前向 == 暴力 log Z')

## ✏️ 练习 3：Viterbi 解码

实现 `viterbi_decode(words, W_emit, W_trans)`：返回 (最优标签名列表, 最优得分)。把前向的 logsumexp 换成 max + 回溯。

In [ ]:
def viterbi_decode(words, W_emit, W_trans):
    # TODO: delta[j]=max_i(delta[i]+trans[i,j])+emit[j]; psi 记 argmax; 回溯
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
path, score = viterbi_decode(words, W_emit, W_trans)
bp, bs = brute_force_best_tags(words, W_emit, W_trans)
assert path == bp and np.allclose(score, bs, atol=1e-10)
assert np.allclose(score, seq_score(words, path, W_emit, W_trans), atol=1e-10)
print('✅ 练习 3 通过：Viterbi == 暴力最优, 得分自洽')

## ✏️ 练习 4：感知机更新

实现 `perceptron_update(We, Wt, words, gold, pred)`：若 `pred != gold`，执行 `We += φ_emit(gold) - φ_emit(pred)`、`Wt += φ_trans(gold) - φ_trans(pred)`，**原地修改** We, Wt 并返回是否更新了（bool）。（用给定的 `feature_counts`。）

In [ ]:
def perceptron_update(We, Wt, words, gold, pred):
    # TODO: gold==pred 返回 False; 否则 We/Wt += φ(gold)-φ(pred), 返回 True
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
We_t = np.zeros((K_t, Vn)); Wt_t = np.zeros((K_t, K_t))
g = ['O', 'B-PER', 'I-PER']; pr = ['O', 'O', 'O']
changed = perceptron_update(We_t, Wt_t, ['mr','john','smith'], g, pr)
assert changed is True
# 更新后, 真值序列得分应比更新前(=0)高
assert seq_score(['mr','john','smith'], g, We_t, Wt_t) > 0
# 预测对了不更新
We2 = np.ones((K_t, Vn)); before = We2.copy()
assert perceptron_update(We2, np.zeros((K_t,K_t)), ['mr','john','smith'], g, g) is False
assert np.array_equal(We2, before), '预测对时不应改权重'
print('✅ 练习 4 通过：感知机更新 (错则奖惩, 对则不动)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def is_capitalized_feature(word):
    return 1 if word[:1].isupper() else 0

def suffix_feature(word, k=3):
    return word[-k:]

In [ ]:
# 练习 2 参考答案
def logZ_forward(words, W_emit, W_trans):
    o = [W2I[w] for w in words]; T = len(o)
    alpha = W_emit[:, o[0]] + W_trans[START, :]
    for t in range(1, T):
        new = np.zeros(K_t)
        for j in range(K_t):
            new[j] = logsumexp(alpha + W_trans[:, j]) + W_emit[j, o[t]]
        alpha = new
    return logsumexp(alpha)

In [ ]:
# 练习 3 参考答案
def viterbi_decode(words, W_emit, W_trans):
    o = [W2I[w] for w in words]; T = len(o)
    delta = W_emit[:, o[0]] + W_trans[START, :]
    psi = np.zeros((T, K_t), dtype=int)
    for t in range(1, T):
        new = np.zeros(K_t)
        for j in range(K_t):
            sc = delta + W_trans[:, j]
            psi[t, j] = int(np.argmax(sc)); new[j] = sc[psi[t, j]] + W_emit[j, o[t]]
        delta = new
    last = int(np.argmax(delta)); path = [last]
    for t in range(T - 1, 0, -1):
        path.append(psi[t, path[-1]])
    path.reverse()
    return [tags[i] for i in path], float(delta[last])

In [ ]:
# 练习 4 参考答案
def perceptron_update(We, Wt, words, gold, pred):
    if gold == pred:
        return False
    fe_g, ft_g = feature_counts(words, gold)
    fe_p, ft_p = feature_counts(words, pred)
    We += fe_g - fe_p; Wt += ft_g - ft_p
    return True

---
## 🧪 真实数据胶囊：CoNLL 风格 NER

用真实风格的 BIO 标注 NER 语料（**内置真实英文 NER 标注句**，CoNLL-2003 同款 BIO 体系），训练结构化感知机，在测试句上解码并算**实体级**准确率。

In [ ]:
def load_ner_data():
    '''CoNLL 风格 BIO 标注句。返回 [([words], [tags]), ...]。'''
    # 真实英文句子 + CoNLL-2003 同款 BIO 标注 (PER 人名, LOC 地名)
    data = [
        (['john', 'smith', 'lives', 'in', 'london'],
         ['B-PER', 'I-PER', 'O', 'O', 'B-LOC']),
        (['mary', 'went', 'to', 'paris'],
         ['B-PER', 'O', 'O', 'B-LOC']),
        (['john', 'visited', 'berlin'],
         ['B-PER', 'O', 'B-LOC']),
        (['mary', 'smith', 'lives', 'in', 'paris'],
         ['B-PER', 'I-PER', 'O', 'O', 'B-LOC']),
        (['john', 'lives', 'in', 'berlin'],
         ['B-PER', 'O', 'O', 'B-LOC']),
        (['mary', 'went', 'to', 'london'],
         ['B-PER', 'O', 'O', 'B-LOC']),
    ]
    return data, 'builtin real CoNLL-style NER'

ner_data, src = load_ner_data()
print('数据来源:', src, '| 句数:', len(ner_data))
# 用胶囊专用变量名(避免覆盖前面的 tags/vocab)
cap_tags = sorted({t for _, ts in ner_data for t in ts})
cap_vocab = sorted({w for ws, _ in ner_data for w in ws})
cap_train, cap_test = ner_data[:4], ner_data[4:]
print('标签:', cap_tags, '| 词表:', len(cap_vocab))
assert len(cap_train) > 0 and len(cap_test) > 0
print('✅ CoNLL 风格 NER 语料就绪')

**🧪 胶囊练习**：用胶囊语料训练结构化感知机并在测试集解码。
这里给出一个**自包含**的训练+解码函数 `train_and_eval_ner`，你只需调用它。补全调用。

In [ ]:
# TODO: acc = train_and_eval_ner(cap_train, cap_test, cap_tags, cap_vocab, epochs=30)
raise NotImplementedError

In [ ]:
# 自测
assert 0.0 <= acc <= 1.0
assert acc >= 0.6, '结构化感知机在规整 NER 上准确率应不低'
print(f'✅ 胶囊练习通过：NER 词级标注准确率 = {acc:.0%}')

In [ ]:
# 📖 胶囊参考答案（自包含: 不依赖前面的全局 tags/vocab/W2I）
def train_and_eval_ner(train, test, tag_list, vocab_list, epochs=30):
    Ti = {t: i for i, t in enumerate(tag_list)}; Kt = len(tag_list)
    Wi = {w: i for i, w in enumerate(vocab_list)}; Vn2 = len(vocab_list)
    ST = Ti.get('O', 0)
    def score_seq(o, y, We, Wt):
        s = We[y[0], o[0]] + Wt[ST, y[0]]
        for t in range(1, len(o)): s += We[y[t], o[t]] + Wt[y[t-1], y[t]]
        return s
    def viterbi(o, We, Wt):
        Tn = len(o); delta = We[:, o[0]] + Wt[ST, :]; psi = np.zeros((Tn, Kt), int)
        for t in range(1, Tn):
            nw = np.zeros(Kt)
            for j in range(Kt):
                sc = delta + Wt[:, j]; psi[t, j] = int(np.argmax(sc)); nw[j] = sc[psi[t, j]] + We[j, o[t]]
            delta = nw
        last = int(np.argmax(delta)); path = [last]
        for t in range(Tn - 1, 0, -1): path.append(psi[t, path[-1]])
        return path[::-1]
    def counts(o, y):
        fe = np.zeros((Kt, Vn2)); ft = np.zeros((Kt, Kt))
        fe[y[0], o[0]] += 1; ft[ST, y[0]] += 1
        for t in range(1, len(o)): fe[y[t], o[t]] += 1; ft[y[t-1], y[t]] += 1
        return fe, ft
    We = np.zeros((Kt, Vn2)); Wt = np.zeros((Kt, Kt))
    Wes = np.zeros_like(We); Wts = np.zeros_like(Wt); n = 0
    for _ in range(epochs):
        for ws, ts in train:
            o = [Wi[w] for w in ws]; g = [Ti[t] for t in ts]
            pr = viterbi(o, We, Wt)
            if pr != g:
                fe_g, ft_g = counts(o, g); fe_p, ft_p = counts(o, pr)
                We += fe_g - fe_p; Wt += ft_g - ft_p
            Wes += We; Wts += Wt; n += 1
    We, Wt = Wes / n, Wts / n
    corr = tot = 0
    for ws, ts in test:
        o = [Wi.get(w, 0) for w in ws]; g = [Ti[t] for t in ts]
        pr = viterbi(o, We, Wt)
        corr += sum(a == b for a, b in zip(pr, g)); tot += len(g)
    return corr / tot

acc = train_and_eval_ner(cap_train, cap_test, cap_tags, cap_vocab, epochs=30)
print('NER 词级准确率 =', round(acc, 3))

### 小结
- **CRF** = 判别式序列模型：建模 $P(S|O)$, 用**特征函数**打分, **全局归一**(配分函数)。
- **特征函数** 可任意/重叠(词/大写/后缀/邻词/词典) —— 这是 CRF 碾压 HMM 的根本。
- **配分函数 $Z(x)$ = CRF 版前向算法**(O(TS²)), **解码 = Viterbi** —— HMM 的 DP 骨架原样复用(sum→max 对偶)。
- **结构化感知机**(Collins): Viterbi 解码 + 错则「真值特征+1, 预测特征-1」, 不需 $Z$, 极简却有效。
- **BIO 编码** 把找实体转成逐词标注; **全局归一**根治 MEMM 的标签偏置。
- 我们对拍**暴力枚举**确认 $\log Z$ 与 Viterbi 精确(1e-10)。

下一站：**模块 05 · NLP 流水线** —— 从序列标注转向文本分类与完整评测(TF-IDF / 朴素贝叶斯 / P/R/F1)。